In [40]:
import pandas as pd
import numpy as np
import pulp

In [41]:
df = pd.read_csv('../data/processed/rq3_inputs.csv', index_col='Datetime (UTC)', parse_dates=True)

In [42]:
BATTERY_CAPACITY = 500
MIN_SOC = 0.1 * BATTERY_CAPACITY
MAX_SOC = 0.9 * BATTERY_CAPACITY
MAX_CHARGE_RATE = 100
MAX_DISCHARGE_RATE = 100
CHARGE_EFFICIENCY = 0.95
DISCHARGE_EFFICIENCY = 0.95
HORIZON = 24

In [43]:
predicted_prices = df['predicted_price'].values
actual_prices = df['actual_price'].values

predicted_P = df['predicted_P'].values
actual_P = df['actual_P'].values

In [44]:
df_test = df.copy()
df_test['date'] = df_test.index.date

In [45]:
daily_stats = df_test.groupby('date').agg(
    price_std=('actual_price', 'std'),
    price_max=('actual_price', 'max'),
    avg_cloud_cover=('cloud_cover', 'mean') if 'cloud_cover' in df_test.columns else ('actual_P', 'mean'),
    is_weekend=('is_weekend', 'first')
)

print(daily_stats)

            price_std  price_max  avg_cloud_cover  is_weekend
date                                                         
2023-09-18  36.764168     157.16        95.777778           0
2023-09-19  40.970344     112.01        79.750000           0
2023-09-20  52.226299     182.28        63.916667           0
2023-09-21  42.334481     238.23        70.958333           0
2023-09-22  24.488167     169.29        74.583333           0
2023-09-23  24.662642     150.59        65.125000           1
2023-09-24  46.537763     147.10        40.875000           1
2023-09-25  66.896859     316.55        60.041667           0
2023-09-26  50.501701     268.13        39.958333           0
2023-09-27  48.150008     270.96        63.291667           0
2023-09-28  41.022332     253.04        84.541667           0
2023-09-29  20.517911     146.55        84.458333           0
2023-09-30  33.301271     162.54        71.958333           1


In [46]:
price_volatility_threshold = daily_stats['price_std'].median()
cloud_threshold = daily_stats['avg_cloud_cover'].median()
price_spike_threshold = daily_stats['price_max'].quantile(0.75)

daily_stats['high_volatility'] = daily_stats['price_std'] > price_volatility_threshold
daily_stats['cloudy'] = daily_stats['avg_cloud_cover'] > cloud_threshold
daily_stats['has_price_spike'] = daily_stats['price_max'] > price_spike_threshold

print(daily_stats)

            price_std  price_max  avg_cloud_cover  is_weekend  \
date                                                            
2023-09-18  36.764168     157.16        95.777778           0   
2023-09-19  40.970344     112.01        79.750000           0   
2023-09-20  52.226299     182.28        63.916667           0   
2023-09-21  42.334481     238.23        70.958333           0   
2023-09-22  24.488167     169.29        74.583333           0   
2023-09-23  24.662642     150.59        65.125000           1   
2023-09-24  46.537763     147.10        40.875000           1   
2023-09-25  66.896859     316.55        60.041667           0   
2023-09-26  50.501701     268.13        39.958333           0   
2023-09-27  48.150008     270.96        63.291667           0   
2023-09-28  41.022332     253.04        84.541667           0   
2023-09-29  20.517911     146.55        84.458333           0   
2023-09-30  33.301271     162.54        71.958333           1   

            high_volatil

In [47]:
def solve_mpc_window(prices, productions, initial_soc, capacity, min_soc, max_soc, max_charge, max_discharge, charge_eff, discharge_eff):

    H = len(prices)
    prob = pulp.LpProblem("battery_mpc", pulp.LpMaximize)

    sell = [pulp.LpVariable(f"sell_{t}", lowBound=0) for t in range(H)]
    charge = [pulp.LpVariable(f"charge_{t}", lowBound=0, upBound=max_charge) for t in range(H)]
    discharge = [pulp.LpVariable(f"discharge_{t}", lowBound=0, upBound=max_discharge) for t in range(H)]
    soc = [pulp.LpVariable(f"soc_{t}", lowBound=min_soc, upBound=max_soc) for t in range(H)]

    prob += pulp.lpSum([prices[t] * (sell[t] + discharge[t] * discharge_eff) / 1000 for t in range(H)])

    for t in range(H):
        prob += sell[t] + charge[t] == productions[t]
        if t == 0:
            prob += soc[t] == initial_soc + charge[t] * charge_eff - discharge[t]
        else:
            prob += soc[t] == soc[t-1] + charge[t] * charge_eff - discharge[t]

    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    return [(sell[t].varValue, charge[t].varValue, discharge[t].varValue) for t in range(H)]

In [48]:
def run_mpc_simulation(predicted_prices, predicted_P, actual_prices, actual_P,
                        capacity, min_soc, max_soc, max_charge, max_discharge,
                        charge_eff, discharge_eff, horizon=24):
    n = len(predicted_prices)
    soc = min_soc
    revenue = 0

    for t in range(n):
        window_end = min(t + horizon, n)
        window_prices = predicted_prices[t:window_end]
        window_prod = predicted_P[t:window_end]

        decisions = solve_mpc_window(
            window_prices, window_prod, soc,
            capacity, min_soc, max_soc,
            max_charge, max_discharge, charge_eff, discharge_eff
        )
        sell_amt, charge_amt, discharge_amt = decisions[0]

        real_price = actual_prices[t]
        real_production = actual_P[t]

        actual_sell = min(sell_amt, real_production)
        actual_charge = min(charge_amt, real_production - actual_sell, max_soc - soc)

        revenue += real_price * (actual_sell + discharge_amt * discharge_eff) / 1000
        soc = soc + actual_charge * charge_eff - discharge_amt

    return revenue

In [49]:
def revenue_for_dates(dates_subset, predicted_prices_full, predicted_P_full,
                       actual_prices_full, actual_P_full, dates_full):
    mask = pd.Series(dates_full).isin(dates_subset).values
    return run_mpc_simulation(
        predicted_prices_full[mask], predicted_P_full[mask],
        actual_prices_full[mask], actual_P_full[mask],
        BATTERY_CAPACITY, MIN_SOC, MAX_SOC,
        MAX_CHARGE_RATE, MAX_DISCHARGE_RATE,
        CHARGE_EFFICIENCY, DISCHARGE_EFFICIENCY, HORIZON
    )

In [50]:
def perturb_random(values, std_pct, seed=None, min_value=0):
    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0, scale=std_pct, size=len(values))
    perturbed = values * (1 + noise)
    return np.clip(perturbed, min_value, None)

In [51]:
def dates_for_regime(regime_mask):
    return daily_stats[regime_mask].index.tolist()

regimes = {
    'high_volatility': dates_for_regime(daily_stats['high_volatility']),
    'low_volatility': dates_for_regime(~daily_stats['high_volatility']),
    'cloudy': dates_for_regime(daily_stats['cloudy']),
    'sunny': dates_for_regime(~daily_stats['cloudy']),
    'weekend': dates_for_regime(daily_stats['is_weekend'] == 1),
    'weekday': dates_for_regime(daily_stats['is_weekend'] == 0),
    'price_spike': dates_for_regime(daily_stats['has_price_spike']),
    'no_spike': dates_for_regime(~daily_stats['has_price_spike']),
}

dates_full = df_test['date'].values

regime_results = {}
for regime_name, dates_subset in regimes.items():
    base_rev = revenue_for_dates(dates_subset, predicted_prices, predicted_P, actual_prices, actual_P, dates_full)
    perturbed_prices = perturb_random(predicted_prices, 0.20, seed=42)
    perturbed_rev = revenue_for_dates(dates_subset, perturbed_prices, predicted_P, actual_prices, actual_P, dates_full)

    n_days = len(dates_subset)
    pct_loss = (perturbed_rev - base_rev) / base_rev * 100 if base_rev != 0 else np.nan

    regime_results[regime_name] = {
        'n_days': n_days,
        'base_revenue': base_rev,
        'perturbed_revenue': perturbed_rev,
        'pct_loss': pct_loss
    }
    print(f"{regime_name} ({n_days} дена): base={base_rev:.2f}, perturbed={perturbed_rev:.2f}, загуба={pct_loss:.2f}%")

high_volatility (6 дена): base=555.01, perturbed=536.76, загуба=-3.29%
low_volatility (7 дена): base=568.21, perturbed=529.60, загуба=-6.79%
cloudy (6 дена): base=476.39, perturbed=448.92, загуба=-5.77%
sunny (7 дена): base=645.58, perturbed=622.57, загуба=-3.56%
weekend (3 дена): base=223.55, perturbed=203.31, загуба=-9.06%
weekday (10 дена): base=886.28, perturbed=864.06, загуба=-2.51%
price_spike (3 дена): base=307.91, perturbed=304.13, загуба=-1.23%
no_spike (10 дена): base=804.66, perturbed=758.08, загуба=-5.79%
